In [13]:
# !pip install opencv-python facenet-pytorch tqdm
# !pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121

In [14]:
import os
import cv2
import torch
from facenet_pytorch import InceptionResnetV1, MTCNN
from tqdm import tqdm
from types import MethodType
import time

In [9]:
import torch

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Torch version: 2.5.1+cu121
CUDA available: True


In [15]:
def encode(img):
    res = resnet(torch.Tensor(img))
    return res
def detect_box(self, img, save_path = None):
    #Detect faces
    batch_boxes, batch_probs, batch_points = self.detect(img, landmarks = True)
    #select faces
    if not self.keep_all:
        batch_boxes, batch_probs, batch_points = self.select_boxes(
            batch_boxes, batch_probs, batch_points, img, method = self.selection_method
        )
    #Extract faces
    faces = self.extract(img, batch_boxes, save_path)
    return batch_boxes, faces
# load model
resnet = InceptionResnetV1(pretrained='vggface2').eval()
mtcnn = MTCNN(
    image_size = 224, keep_all = True, thresholds = [0.4, 0.5, 0.5], min_face_size = 60
)
mtcnn.detect_box = MethodType(detect_box, mtcnn)

In [29]:
# get encoded features for all saved images
BASE_DIR = os.getcwd()
saved_pictures = os.path.join(BASE_DIR, "Saved")
#saved_pictures = (r"C:\Users\Archit\Documents\ML Projects\Humanoid Robot\Face_Recognnition\Saved")
all_people_faces = {}

In [30]:
BASE_DIR

'C:\\Users\\Archit\\Documents\\ML Projects\\Humanoid Robot\\Face_Recognnition'

In [31]:
saved_pictures

'C:\\Users\\Archit\\Documents\\ML Projects\\Humanoid Robot\\Face_Recognnition\\Saved'

In [32]:
'''for file in person_face, extension = file.split(".")
    img = cv2.imread(f"{saved_pictures}/{person_face}.jpg")
    cropped = mtcnn(img)
    if cropped is not None:
        all_people_faces[person_face] = encode(cropped)[0, :1]'''
for file in os.listdir(saved_pictures):
    if file.endswith('.jpg') or file.endswith('.png'):
        person_face = os.path.splitext(file)[0]
        image_path = os.path.join(saved_pictures, file)

        img = cv2.imread(image_path)
        if img is None:
            continue 
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        cropped = mtcnn(img_rgb)

        if cropped is not None:
            if len(cropped.shape) == 3:
                cropped = cropped.unsqueeze(0) # add batch dim

            embedding = encode (cropped).detach()
            all_people_faces[person_face] = embedding

In [33]:
# all_people_faces

In [34]:
def detect(cam=0, thres=0.7):
    # Initialize video capture (default webcam = 0)
    vdo = cv2.VideoCapture(cam)

    # For FPS calculation
    prev_time = time.time()
    
    while True:
        # Read frame from camera
        ret, img0 = vdo.read()
        if not ret:
            break

        # Detect faces and extract cropped face images
        batch_boxes, cropped_images = mtcnn.detect_box(img0)
    
        if cropped_images is not None:
            for box, cropped in zip(batch_boxes, cropped_images):
                # Extract bounding box coordinates
                x, y, x2, y2 = [int(x) for x in box]

                # Ensure correct shape for embedding model
                if len(cropped.shape) == 3:
                    cropped = cropped.unsqueeze(0)

                # Generate embedding for detected face
                img_embedding = encode(cropped)
                
                # Compare with known face embeddings
                detect_dict = {}
                for name, embedding in all_people_faces.items():
                    detect_dict[name] = (embedding - img_embedding).norm().item()

                # Find closest match
                min_key = min(detect_dict, key=detect_dict.get)

                # Apply threshold to decide if recognized or not
                if detect_dict[min_key] >= thres:
                    min_key = 'Undetected'
    
                # Draw bounding box and label
                cv2.rectangle(img0, (x, y), (x2, y2), (0, 0, 255), 2)
                cv2.putText(
                    img0, min_key, (x + 5, y + 15),
                    cv2.FONT_HERSHEY_DUPLEX, 0.5, (255, 255, 255), 1
                )
    
        # ===== FPS Calculation =====
        curr_time = time.time()
        fps = 1 / (curr_time - prev_time)
        prev_time = curr_time

        # Display FPS on screen
        cv2.putText(
            img0, f"FPS: {fps:.2f}", (10, 30),
            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2
        )

        # Show output frame
        cv2.imshow('output', img0)

        # Exit loop when 'q' is pressed
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    # Release camera and close all OpenCV windows
    vdo.release()
    cv2.destroyAllWindows()

In [35]:
if __name__ == '__main__':
    detect(0)